# Next development step: paired ordinary and stored INT4 caches

Person A's week-one baseline has been verified. This is the next development experiment, using the same 80 exposed questions and three pinned checkpoints. Both conditions run on the same loaded FP32 model. Total: **480 response records**, plus reference scores on the 20 forget questions per model/condition.

INT4 uses HQQ, axes 1/1, group64 and residual128. The first forget item in each model has a traced consumption/neutrality probe and a native-generation trace. All positive, negative and unchanged outcomes are retained. This is limited development evidence; independent review and confirmation choices remain open. No new confirmation author is accessed.

## 1. Verify GPU selection

In [ ]:
import subprocess,sys
print(sys.version)
subprocess.run(["nvidia-smi"],check=True)


## 2. Set up packages
Run this cell. It skips installation when the declared pins already match. In a new default runtime it installs B/day-six versions. If installation occurs, **restart the session**, then continue at section 3. Removing torchvision/torchaudio avoids the import conflict observed by B. No datasets package is needed for frozen JSON inputs.

In [ ]:
import importlib.metadata,subprocess,sys
FORCE_SETUP=False
PINS={"torch":"2.9.1","transformers":"5.5.4","hqq":"0.2.8.post1","accelerate":"1.13.0","huggingface-hub":"1.33.0","numpy":"2.1.3"}
def matches(name,expected):
    try:return importlib.metadata.version(name).split("+")[0]==expected
    except importlib.metadata.PackageNotFoundError:return False
if FORCE_SETUP or not all(matches(k,v) for k,v in PINS.items()):
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchvision","torchaudio"],check=True)
    subprocess.run([sys.executable,"-m","pip","install","torch==2.9.1","--index-url","https://download.pytorch.org/whl/cu128"],check=True)
    subprocess.run([sys.executable,"-m","pip","install"]+[k+"=="+v for k,v in PINS.items() if k!="torch"],check=True)
    print("Setup completed. RESTART the session, then continue at section 3.")
else:print("Declared versions already match. Continue at section 3.")


## 3. Verify the environment after any required restart

In [ ]:
import torch,transformers,importlib.metadata
assert torch.cuda.is_available()
assert torch.__version__.split("+")[0]=="2.9.1" and torch.version.cuda=="12.8", "Use the CUDA 12.8 wheel in setup."
assert transformers.__version__=="5.5.4"
assert importlib.metadata.version("hqq")=="0.2.8.post1"
print(torch.cuda.get_device_name(0),torch.__version__,transformers.__version__)


## 4. Upload the paired-development ZIP
Upload `int4-development-input-bundle.zip`. Leave RESUME_OUTPUT_DIR blank for a new run. A continuation requires the exact existing output-directory path, matching bundle/environment, and intact successful prior rows. Failed or malformed records are preserved and require investigation. Download partial artifacts before ending a temporary runtime.

In [ ]:
from google.colab import files
from pathlib import Path
from datetime import datetime,timezone
import hashlib,zipfile,io,json,sys
RESUME_OUTPUT_DIR=""
uploaded=files.upload()
NAME="int4-development-input-bundle.zip"
assert NAME in uploaded
payload=uploaded[NAME]
assert hashlib.sha256(payload).hexdigest()=="7a4a675726906acf9da1eb81c9c85c0360e3184cfc7b755f44b0dbc16c3e3e63", "Notebook and bundle differ."
BASE=Path("/content/kv-int4-development");BASE.mkdir(parents=True,exist_ok=True)
STAMP=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
SOURCE=BASE/("input_"+STAMP);SOURCE.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        assert Path(name).name==name,"Unexpected archive path."
        (SOURCE/name).write_bytes(z.read(name))
PREP=json.loads((SOURCE/"preparation-manifest.json").read_text())
for name,digest in PREP["files"].items():assert hashlib.sha256((SOURCE/name).read_bytes()).hexdigest()==digest
OUT=Path(RESUME_OUTPUT_DIR) if RESUME_OUTPUT_DIR else BASE/("int4dev_"+STAMP)
sys.path.insert(0,str(SOURCE))
print("Output directory:",OUT)


## 5. Run the matched conditions
Each model loads once. The per-model INT4 trace must pass before the main loop. Ordinary rows are regenerated and compared with day seven; INT4 rows are paired with current ordinary rows. Utility is included in both conditions. Results append after each condition/item.

A quantized sequential score is allowed to differ from its parallel score. Control gates check ordinary agreement, empty-cache prefill, first-target behavior and cache consumption. They do not require recovery or changed answers. If this cell fails, run section 6 to preserve the failure and partial evidence.

In [ ]:
import importlib.util,traceback
spec=importlib.util.spec_from_file_location("kv_int4_development_runner",SOURCE/"int4_development.py")
runner=importlib.util.module_from_spec(spec)
try:
    spec.loader.exec_module(runner)
    report=runner.main(SOURCE,OUT,resume=bool(RESUME_OUTPUT_DIR))
    print("STATUS:",report["status"])
except Exception:
    OUT.mkdir(parents=True,exist_ok=True)
    (OUT/"notebook-error.txt").write_text(traceback.format_exc())
    raise
finally:
    if OUT.exists():(OUT/"input-bundle-sha256.txt").write_text(hashlib.sha256(payload).hexdigest()+"\n")


## 6. Download output, including partial evidence
Return the ZIP for verification and proposed paired factual/utility annotation. Do not infer correctness from changed text or probabilities alone. Preserve truncations and errors. No labels are automatically assigned by this notebook.

In [ ]:
import shutil
assert OUT.exists()
archive=shutil.make_archive(str(OUT),"zip",root_dir=OUT)
print(archive)
files.download(archive)
